In [ ]:
def main(datasources, start_date, end_date):
    """
    selected_tiny_tf_a31_v1 · A31 平台自洽提交包（吸取 v11 / a11_v2 通路）

    消融 A31：A1 宽结构 × 去近复制 fin（33→30）
      drop: fin_pv_cand12 / fin_e22_v5 / fin_pv_cand06
      keep: 全部 29 PV + fin_pv_cand09
    结构：d_model=128, dim_ff=256, n_layers=2, seq_len=20；NTILE10 CE + Softmax 加权读出

    【平台稳健（对齐 v11 / a11_v2）】
      - 单 notebook；禁止本地 parquet / feat_dump / abl_runtime
      - 日频 SQL：first/last/SUM/AVG/ARG_* + CASE；禁止 LAG/MEDIAN OVER / FILTER / rn_desc
      - 尾盘：墙钟 hm>=1430；按月分片；空月 try/except 跳过
      - 训窗写死 2023（优先能跑完）；lab A31 更长窗仅本地
      - 训练写死 bigalpha_2026_*；预测用 datasources；特征缺失 fillna(0.5)

    返回: date, instrument, factor
    """
    import gc
    import time
    from concurrent.futures import ThreadPoolExecutor
    from typing import List

    import numpy as np
    import pandas as pd
    import dai
    import structlog
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset, ConcatDataset

    logger = structlog.get_logger()

    # 平台稳健训窗：对齐 v11（全年 2023）；勿拉到 lab 的 2022H1~2023H1
    TRAIN_START = "2023-01-01 00:00:00"
    TRAIN_END = "2023-12-31 23:59:59"
    TRAIN_BAR1M = "bigalpha_2026_stock_bar1m"
    TRAIN_FIN = "bigalpha_2026_financial"

    FEATURE_COLS = [
        "c17",
        "ideation_h_84",
        "pv_postwk_spec",
        "pv_sal_up",
        "probe_d_12",
        "pv_left_tail",
        "pv_tail_frag",
        "ideation_q_100",
        "pv_vol_shock_on",
        "pv_min_chase",
        "ideation_n_72",
        "ideation_n_72_v6",
        "ideation_g_86",
        "c25",
        "pv_tug_ticks",
        "pv_hit_fat_ask",
        "pv_close_deal_imb",
        "pv_fip_cont",
        "pv_resid_rev",
        "pv_spr_widen_on_flow",
        "pv_preclose_rush",
        "ideation_m_90",
        "pv_cgo_neg",
        "pv_vol_ul_shape_v3",
        "b97",
        "ideation_m_88",
        "pv_frag_shock",
        "pv_signed_deal",
        "pv_deal_amihud",
        "fin_pv_cand09",
    ]
    INPUT_SIZE = len(FEATURE_COLS)
    SEQ_LEN = 20
    NUM_CLASSES = 10
    D_MODEL, NHEAD, N_LAYERS, DIM_FF = 128, 4, 2, 256
    DROPOUT = 0.05
    NUM_EPOCHS = 12
    LR = 1e-4
    BATCH_SIZE = 2048
    SEED = 42
    MIN_SAMPLE = 38
    LOOKBACK_DAYS = 120 * 3 + 80  # 对齐 v11；覆盖 MA60 + 序列缓冲
    CHUNK_MONTHS = 1

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    def _ntile10(s):
        n = int(s.notna().sum())
        if n < NUM_CLASSES:
            return pd.Series(np.nan, index=s.index)
        r = s.rank(method="first")
        out = np.floor((r - 1.0) / n * NUM_CLASSES).clip(0, NUM_CLASSES - 1)
        return pd.Series(out, index=s.index)

    def _cs_rank(s: pd.Series, min_sample: int = MIN_SAMPLE) -> pd.Series:
        if int(s.notna().sum()) < min_sample:
            return pd.Series(np.nan, index=s.index)
        return s.rank(pct=True)


    def _cs_zscore(s: pd.Series, min_sample: int = MIN_SAMPLE) -> pd.Series:
        if int(s.notna().sum()) < min_sample:
            return pd.Series(np.nan, index=s.index)
        mu, sd = s.mean(), s.std(ddof=0)
        if sd is None or not np.isfinite(sd) or sd < 1e-12:
            return pd.Series(np.nan, index=s.index)
        return (s - mu) / sd


    def _mat_roll_rank(s: pd.Series, d: int) -> pd.Series:
        return s.rolling(d, min_periods=d).apply(
            lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
        )


    def _ts_beta_r2(y: pd.Series, x: pd.Series, d: int):
        mp = max(2, d // 2)
        cov = y.rolling(d, min_periods=mp).cov(x)
        var = x.rolling(d, min_periods=mp).var()
        beta = cov / var.replace(0, np.nan)
        corr = y.rolling(d, min_periods=mp).corr(x)
        r2 = (corr ** 2).clip(0.0, 1.0)
        return beta, r2


    def _month_chunks(sd: pd.Timestamp, ed: pd.Timestamp, months: int = 1):
        cur = sd.normalize().replace(day=1)
        end = ed.normalize()
        while cur <= end:
            nxt = cur + pd.DateOffset(months=months)
            c0 = max(cur, sd.normalize())
            c1 = min(nxt - pd.Timedelta(seconds=1), ed)
            if c0 <= c1:
                yield c0, c1
            cur = nxt


    def _fat_sql(bar1m: str) -> str:
        """v11 方言日频胖面板：无 LAG/MEDIAN OVER / FILTER；尾盘墙钟 hm>=1430。"""
        buy_w = (
            "(COALESCE(bid_volume1,0)*1.0+COALESCE(bid_volume2,0)*EXP(-0.3)"
            "+COALESCE(bid_volume3,0)*EXP(-0.6)+COALESCE(bid_volume4,0)*EXP(-0.9)"
            "+COALESCE(bid_volume5,0)*EXP(-1.2))"
        )
        ask_w = (
            "(COALESCE(ask_volume1,0)*1.0+COALESCE(ask_volume2,0)*EXP(-0.3)"
            "+COALESCE(ask_volume3,0)*EXP(-0.6)+COALESCE(ask_volume4,0)*EXP(-0.9)"
            "+COALESCE(ask_volume5,0)*EXP(-1.2))"
        )
        buy_share = f"({buy_w} / NULLIF({buy_w} + {ask_w}, 0))"
        bid_n = (
            "(COALESCE(bid_num_orders1,0)+COALESCE(bid_num_orders2,0)"
            "+COALESCE(bid_num_orders3,0)+COALESCE(bid_num_orders4,0)"
            "+COALESCE(bid_num_orders5,0))"
        )
        conc = f"(COALESCE(bid_num_orders1,0)*1.0 / NULLIF({bid_n}, 0))"
        large = "deal_number > 0 AND amount / deal_number BETWEEN 200000 AND 1000000"
        mid = "deal_number > 0 AND amount / deal_number BETWEEN 40000 AND 200000"
        voi1 = (
            "((COALESCE(bid_volume1,0)-COALESCE(ask_volume1,0))"
            "/NULLIF(COALESCE(bid_volume1,0)+COALESCE(ask_volume1,0),0))"
        )
        oir = f"(({buy_w})-({ask_w}))/NULLIF(({buy_w})+({ask_w})+1e-8,0)"
        mid_px = "((COALESCE(bid_price1,0)+COALESCE(ask_price1,0))/2.0)"
        ask_amt = (
            "(COALESCE(ask_price1,0)*COALESCE(ask_volume1,0)+COALESCE(ask_price2,0)*COALESCE(ask_volume2,0)"
            "+COALESCE(ask_price3,0)*COALESCE(ask_volume3,0)+COALESCE(ask_price4,0)*COALESCE(ask_volume4,0)"
            "+COALESCE(ask_price5,0)*COALESCE(ask_volume5,0))"
        )
        ask_vol = (
            "(COALESCE(ask_volume1,0)+COALESCE(ask_volume2,0)+COALESCE(ask_volume3,0)"
            "+COALESCE(ask_volume4,0)+COALESCE(ask_volume5,0))"
        )
        mci = f"((({ask_amt})/NULLIF({ask_vol},0)-({mid_px}))/NULLIF({mid_px},0))/NULLIF({ask_vol},0)"
        mofi = (
            "("
            "1.0*((COALESCE(bid_volume1,0))-(COALESCE(ask_volume1,0)))/NULLIF((COALESCE(bid_volume1,0))+(COALESCE(ask_volume1,0)),0)"
            "+2.0*((COALESCE(bid_volume2,0))-(COALESCE(ask_volume2,0)))/NULLIF((COALESCE(bid_volume2,0))+(COALESCE(ask_volume2,0)),0)"
            "+3.0*((COALESCE(bid_volume3,0))-(COALESCE(ask_volume3,0)))/NULLIF((COALESCE(bid_volume3,0))+(COALESCE(ask_volume3,0)),0)"
            "+4.0*((COALESCE(bid_volume4,0))-(COALESCE(ask_volume4,0)))/NULLIF((COALESCE(bid_volume4,0))+(COALESCE(ask_volume4,0)),0)"
            "+5.0*((COALESCE(bid_volume5,0))-(COALESCE(ask_volume5,0)))/NULLIF((COALESCE(bid_volume5,0))+(COALESCE(ask_volume5,0)),0)"
            ")/15.0"
        )
        hm = "(EXTRACT(hour FROM date)*100 + EXTRACT(minute FROM date))"
        spr = f"((ask_price1-bid_price1)/NULLIF({mid_px},0))"
        is_tail = f"({hm} >= 1430)"
        is_pre = f"({hm} >= 1430 AND {hm} < 1457)"
        is_open = f"({hm} BETWEEN 930 AND 1000)"
        tot_bid = (
            "(COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0)"
            "+COALESCE(bid_volume4,0)+COALESCE(bid_volume5,0))"
        )
        tot_ask = ask_vol
        # 无分钟 LAG：用买盘份额代理 signed / close_deal 失衡
        signed_w = f"(2.0 * COALESCE({buy_share}, 0.5) - 1.0)"

        return f"""
        SELECT
            date_trunc('day', date)::DATE AS trading_day,
            instrument,
            SUM(volume) AS volume,
            SUM(amount) AS amount,
            SUM(deal_number) AS deal_number,
            first(open ORDER BY date) AS day_open,
            MAX(high) AS day_high,
            MIN(low) AS day_low,
            last(close ORDER BY date) AS day_close,
            first(pre_close ORDER BY date) AS pre_close,
            COUNT(*) AS n_min,
            SUM(amount * COALESCE({buy_share}, 0.5)) AS b_amount,
            SUM(amount * (1.0 - COALESCE({buy_share}, 0.5))) AS s_amount,
            SUM(amount * (1.0 - COALESCE({buy_share}, 0.5))) AS cs_amount,
            SUM(deal_number * COALESCE({buy_share}, 0.5)) AS b_deals,
            SUM(CASE WHEN {large} THEN amount * COALESCE({buy_share}, 0.5) ELSE 0 END) AS b_amount_l,
            SUM(CASE WHEN {large} THEN amount * (1.0 - COALESCE({buy_share}, 0.5)) ELSE 0 END) AS s_amount_l,
            SUM(CASE WHEN {mid} THEN amount * COALESCE({buy_share}, 0.5) ELSE 0 END) AS b_amount_m,
            AVG(CASE WHEN {mid} THEN {conc} END) AS b_concentration_m,
            AVG({conc}) AS b_concentration,
            ARG_MAX({voi1}, date) AS voi1_last,
            AVG({oir}) AS oir_mean,
            AVG({mci}) AS mci_a_mean,
            AVG({tot_bid}) AS tot_bid_mean,
            AVG({tot_ask}) AS tot_ask_mean,
            AVG({mofi}) AS mofi_imb_mean,
            SUM(COALESCE(ask_volume1, 0)) AS ask_v1,
            SUM(COALESCE(ask_num_orders1, 0)) AS ask_n1,
            AVG(CASE WHEN ask_price1 > bid_price1 THEN {spr} END) AS spr,
            SUM(CASE WHEN {is_open} THEN amount ELSE 0 END) AS amt_open,
            SUM(CASE WHEN {is_pre} THEN amount ELSE 0 END) AS amt_pre,
            ARG_MIN(close, CASE WHEN {is_pre} THEN date END) AS px_pre_open,
            ARG_MAX(close, CASE WHEN {is_pre} THEN date END) AS px_pre_last,
            SUM(amount * {signed_w}) / NULLIF(SUM(amount), 0) AS signed_deal_imb,
            SUM(CASE WHEN {is_tail} THEN amount * {signed_w} ELSE 0 END)
                / NULLIF(SUM(CASE WHEN {is_tail} THEN amount ELSE 0 END), 0) AS close_deal_imb,
            last(close ORDER BY date) AS close_px,
            ARG_MIN(open, CASE WHEN {is_tail} THEN date END) AS tail_start_px,
            SUM(CASE WHEN {is_tail} THEN amount ELSE 0 END) AS amt_tail,
            SUM(CASE WHEN {is_tail} THEN deal_number ELSE 0 END) AS deal_tail,
            AVG({spr}) AS spr_hi,
            AVG({spr}) AS spr_lo
        FROM {bar1m}
        WHERE ask_price1 > 0 AND bid_price1 > 0
          AND open > 0 AND high > 0 AND low > 0 AND close > 0
        GROUP BY date_trunc('day', date), instrument
        ORDER BY trading_day, instrument
        """


    def _query_month(sql, c0, c1):
        """v11：单月 query；失败或空表返回空 DataFrame，不中断整段。"""
        s = pd.Timestamp(c0).strftime("%Y-%m-%d 00:00:00")
        e = pd.Timestamp(c1).strftime("%Y-%m-%d 23:59:59")
        try:
            df = dai.query(sql, filters={"date": [s, e]}, compression=True).df()
            if df is not None and len(df) > 0:
                return df
        except Exception:
            pass
        return pd.DataFrame()

    def query_fat_panel(bar1m: str, start_date, end_date) -> pd.DataFrame:
        """按月分片 + 空月跳过（v11 _query_month）。"""
        sd = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)
        ed = pd.to_datetime(end_date)
        sql = _fat_sql(bar1m)
        parts: List[pd.DataFrame] = []
        t0 = time.time()
        for i, (c0, c1) in enumerate(_month_chunks(sd, ed, CHUNK_MONTHS), 1):
            chunk = _query_month(sql, c0, c1)
            if chunk is None or chunk.empty:
                continue
            parts.append(chunk)
            if i % 6 == 0:
                logger.info("fat panel chunk", i=i, rows=len(chunk), sec=round(time.time() - t0, 1))
        if not parts:
            return pd.DataFrame()
        df = pd.concat(parts, ignore_index=True)
        del parts
        gc.collect()
        df["date"] = pd.to_datetime(df["trading_day"]).dt.normalize()
        df["instrument"] = df["instrument"].astype(str)
        num_cols = [c for c in df.columns if c not in ("trading_day", "date", "instrument")]
        for c in num_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df.drop_duplicates(["date", "instrument"], keep="last").sort_values(
            ["instrument", "date"]
        ).reset_index(drop=True)
        df["bs_rate"] = (df["b_amount"] - df["s_amount"]) / (df["b_amount"] + df["s_amount"]).replace(0, np.nan)
        df["open_share"] = df["amt_open"] / df["amount"].replace(0, np.nan)
        df["amt_day"] = df["amount"]
        df["deal_day"] = df["deal_number"]
        miss_tail = float(df["tail_start_px"].isna().mean()) if "tail_start_px" in df.columns else 1.0
        if miss_tail > 0.8:
            df["tail_start_px"] = df["day_open"]
            logger.warning("tail_start_px 缺失过多，退化为 day_open", miss=round(miss_tail, 3))
        logger.info("fat panel ready", rows=len(df), days=int(df["date"].nunique()), sec=round(time.time() - t0, 1))
        return df


    def query_fin_panel(financial: str, start_date, end_date) -> pd.DataFrame:
        qsd = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)
        sql = f"""
        SELECT date, instrument, shift, category,
               net_cffoa, moneytary_assets, accounts_receivable, total_assets
        FROM {financial}
        WHERE category IN ('ttm', 'lf')
        """
        fin = dai.query(
            sql,
            filters={"date": [qsd.strftime("%Y-%m-%d %H:%M:%S"), end_date]},
            compression=True,
        ).df()
        fin["date"] = pd.to_datetime(fin["date"]).dt.normalize()
        fin["instrument"] = fin["instrument"].astype(str)
        for c in ("net_cffoa", "moneytary_assets", "accounts_receivable", "total_assets", "shift"):
            if c in fin.columns:
                fin[c] = pd.to_numeric(fin[c], errors="coerce")
        return fin


    def _sparse_pit_ffill(cal: pd.DataFrame, fin: pd.DataFrame, cols: List[str], category: str) -> pd.DataFrame:
        """PIT: sparse shift=0 events → calendar ffill (no daily shift YoY)."""
        sub = fin[fin["category"] == category].copy()
        if "shift" in sub.columns:
            sub = sub.sort_values(["instrument", "date", "shift"])
            sub = sub.groupby(["instrument", "date"], as_index=False).first()
        keep = ["date", "instrument"] + [c for c in cols if c in sub.columns]
        fin_s = sub[keep].copy()
        fin_s["_src"] = 0
        cal_s = cal[["date", "instrument"]].drop_duplicates().copy()
        cal_s["_src"] = 1
        both = pd.concat([fin_s, cal_s], ignore_index=True, sort=False)
        both = both.sort_values(["instrument", "date", "_src"]).reset_index(drop=True)
        both[cols] = both.groupby("instrument", sort=False)[cols].ffill()
        out = both[both["_src"] == 1].drop(columns=["_src"])
        return out


    def query_fltcap(start_date, end_date) -> pd.DataFrame:
        qsd = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)
        flib = dai.query(
            "SELECT date, instrument, float_market_cap FROM bigalpha_2026_factorlib",
            filters={"date": [qsd.strftime("%Y-%m-%d %H:%M:%S"), end_date]},
        ).df()
        flib["date"] = pd.to_datetime(flib["date"]).dt.normalize()
        flib["instrument"] = flib["instrument"].astype(str)
        flib["float_market_cap"] = pd.to_numeric(flib["float_market_cap"], errors="coerce")
        return flib[["date", "instrument", "float_market_cap"]]


    # ---------------- 33 pandas legs (panel -> Series factor) ----------------

    def _g(df: pd.DataFrame):
        return df.groupby("instrument", sort=False)


    def _attach_cs(df: pd.DataFrame, raw: pd.Series, neg: bool = False) -> pd.Series:
        tmp = df[["date"]].copy()
        tmp["_r"] = -raw if neg else raw
        return tmp.groupby("date")["_r"].transform(_cs_rank)


    def compute_all_legs(panel: pd.DataFrame) -> pd.DataFrame:
        """Apply 33 pandas recipes; return date,instrument + FEATURE_COLS."""
        df = panel.sort_values(["instrument", "date"]).reset_index(drop=True)
        g = df.groupby("instrument", sort=False)
        out = df[["date", "instrument"]].copy()
        EPS = 1e-12

        # --- simple ---
        a_pt = df["amount"] / df["deal_number"].replace(0, np.nan)
        out["c17"] = _attach_cs(df, a_pt, neg=True)

        net_l = df["b_amount_l"] - df["s_amount_l"]
        bl = np.where(df["b_amount_l"] > 0, df["b_amount_l"], df["b_amount"])
        out["ideation_h_84"] = (
            df.assign(_n=net_l, _b=bl)
            .groupby("date")["_n"].transform(_cs_rank)
            - df.assign(_b=bl).groupby("date")["_b"].transform(_cs_rank)
        )

        # postwk
        rng = (df["day_high"] - df["day_low"]) / df["day_close"].replace(0, np.nan)
        amt_ma20 = g["amount"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        rel = df["amount"] / amt_ma20.replace(0, np.nan)
        rng_ma = rng.groupby(df["instrument"]).transform(lambda s: s.rolling(5, min_periods=3).mean())
        rel_ma = rel.groupby(df["instrument"]).transform(lambda s: s.rolling(5, min_periods=3).mean())
        spec = rng_ma * rel_ma
        dgap = g["date"].diff().dt.days
        postwk = dgap >= 3
        spec_lag = spec.groupby(df["instrument"]).shift(1)
        raw_pw = np.where(postwk.to_numpy(), spec_lag.to_numpy(), spec.to_numpy())
        out["pv_postwk_spec"] = _attach_cs(df, pd.Series(raw_pw, index=df.index), neg=False)

        # sal_up: salience of up moves over L=20
        ret = g["day_close"].pct_change()
        cs_mean = ret.groupby(df["date"]).transform("mean")
        diff = ret - cs_mean
        abs_diff = diff.abs()
        cs_abs = abs_diff.groupby(df["date"]).transform("mean")
        sal = abs_diff / (cs_abs + EPS)
        sal_up = np.where(diff > 0, sal, 0.0)
        sal_up = pd.Series(sal_up, index=df.index)
        num = sal_up.groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).sum())
        den = sal.groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).sum())
        st_up = num / (den + EPS)
        out["pv_sal_up"] = _attach_cs(df, st_up, neg=True)

        b_apt = df["b_amount"] / df["b_deals"].replace(0, np.nan)
        out["probe_d_12"] = (
            df.groupby("date")["voi1_last"].transform(_cs_rank)
            - df.assign(_x=b_apt).groupby("date")["_x"].transform(_cs_rank)
        )

        # left_tail
        q05 = ret.groupby(df["instrument"]).transform(lambda s: s.rolling(60, min_periods=30).quantile(0.05))
        # var5=-q05(ret,60); factor=CSRank(-var5)=CSRank(q05)
        out["pv_left_tail"] = _attach_cs(df, q05, neg=False)

        # tail_frag (wall-clock tail)
        tail_ret = (df["close_px"] - df["tail_start_px"]) / df["tail_start_px"].replace(0, np.nan)
        amt_part = df["amt_tail"] / (df["amt_day"] + EPS)
        deal_part = df["deal_tail"] / (df["deal_day"] + EPS)
        mix = deal_part / (amt_part + EPS)
        out["pv_tail_frag"] = _attach_cs(df, tail_ret * mix, neg=True)

        # q_100
        if "float_market_cap" in df.columns:
            flt = df["float_market_cap"]
        else:
            flt = pd.Series(np.nan, index=df.index)
        out["ideation_q_100"] = (
            df.assign(_a=a_pt).groupby("date")["_a"].transform(_cs_rank)
            - df.assign(_f=flt).groupby("date")["_f"].transform(_cs_rank)
        )
        out["ideation_q_100"] = -out["ideation_q_100"]

        # vol_shock
        ma60 = g["amount"].transform(lambda s: s.rolling(60, min_periods=30).mean()).groupby(df["instrument"]).shift(1)
        shock = df["amount"] / ma60.replace(0, np.nan) - 1.0
        out["pv_vol_shock_on"] = _attach_cs(df, shock, neg=True)

        # min_chase
        rmin = ret.groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).min())
        out["pv_min_chase"] = _attach_cs(df, rmin, neg=False)

        # n72
        beta20, _ = _ts_beta_r2(df["cs_amount"], df["bs_rate"], 20)
        # need per-instrument rolling — apply via groupby
        def _beta_by_inst(ycol, xcol, d):
            betas = []
            for _, sub in df.groupby("instrument", sort=False):
                b, r2 = _ts_beta_r2(sub[ycol], sub[xcol], d)
                betas.append(b)
            return pd.concat(betas).reindex(df.index)

        beta20 = _beta_by_inst("cs_amount", "bs_rate", 20)
        out["ideation_n_72"] = _attach_cs(df, beta20, neg=False)

        beta5, r2_5 = [], []
        for _, sub in df.groupby("instrument", sort=False):
            b, r2 = _ts_beta_r2(sub["cs_amount"], sub["bs_rate"], 5)
            beta5.append(b)
            r2_5.append(r2)
        beta5 = pd.concat(beta5).reindex(df.index)
        r2_5 = pd.concat(r2_5).reindex(df.index)
        score = beta5 * (0.30 + 0.70 * r2_5)
        size = np.log1p(df["amount"].clip(lower=0))
        out["ideation_n_72_v6"] = (
            _attach_cs(df, score, neg=False) - 0.08 * _attach_cs(df, size, neg=False)
        )

        # g_86
        amt_m = np.where(df["b_amount_m"].notna() & (df["b_amount_m"] > 0), df["b_amount_m"], df["b_amount"])
        conc_m = np.where(df["b_concentration_m"].notna(), df["b_concentration_m"], df["b_concentration"])
        out["ideation_g_86"] = (
            df.assign(_c=conc_m).groupby("date")["_c"].transform(_cs_rank)
            - df.assign(_a=amt_m).groupby("date")["_a"].transform(_cs_rank)
        )

        # c25 MatRollRank
        roll = a_pt.groupby(df["instrument"]).transform(
            lambda s: s.rolling(20, min_periods=10).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
        )
        out["c25"] = -roll  # already ts-rank; keep as factor scale (notebook -MatRollRank)

        # tug
        on = df["day_open"] / df["pre_close"].replace(0, np.nan) - 1.0
        intrad = df["day_close"] / df["day_open"].replace(0, np.nan) - 1.0
        tug = (
            on.groupby(df["instrument"]).transform(lambda s: s.rolling(40, min_periods=20).mean())
            - intrad.groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).mean())
        )
        out["pv_tug_ticks"] = _attach_cs(df, tug / (df["spr"] + EPS), neg=False)

        # hit_fat_ask
        aos = df["ask_v1"] / df["ask_n1"].replace(0, np.nan)
        aos_ma = aos.groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).mean())
        z = aos / aos_ma.replace(0, np.nan)
        up = (df["day_close"] / df["day_open"].replace(0, np.nan) - 1.0).clip(lower=0)
        out["pv_hit_fat_ask"] = _attach_cs(df, up * z, neg=True)

        out["pv_close_deal_imb"] = _attach_cs(df, df["close_deal_imb"], neg=True)

        # fip
        prev = g["day_close"].shift(1)
        r = df["day_close"] / prev.replace(0, np.nan) - 1.0
        pret = (1.0 + r).groupby(df["instrument"]).transform(
            lambda s: s.rolling(20, min_periods=10).apply(lambda x: np.nanprod(x) - 1.0, raw=True)
        )
        f_pos = (r > 0).astype(float).groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).mean())
        f_neg = (r < 0).astype(float).groupby(df["instrument"]).transform(lambda s: s.rolling(20, min_periods=10).mean())
        id_ = np.sign(pret) * (f_neg - f_pos)
        out["pv_fip_cont"] = _attach_cs(df, pret * (-id_), neg=True)

        # resid_rev
        cs_m = r.groupby(df["date"]).transform("mean")
        resid = r - cs_m
        ma5 = resid.groupby(df["instrument"]).transform(lambda s: s.rolling(5, min_periods=3).mean())
        out["pv_resid_rev"] = _attach_cs(df, -ma5, neg=False)

        # spr_widen
        widen = df["spr_hi"] / df["spr_lo"].replace(0, np.nan)
        day_ret = df["day_close"] / df["day_open"].replace(0, np.nan) - 1.0
        out["pv_spr_widen_on_flow"] = _attach_cs(df, np.sign(day_ret) * widen, neg=True)

        # preclose
        rush = (df["amt_pre"] / df["amount"].replace(0, np.nan)) * (
            df["px_pre_last"] / df["px_pre_open"].replace(0, np.nan) - 1.0
        )
        out["pv_preclose_rush"] = _attach_cs(df, rush, neg=True)

        # m_90
        def _mr(col, d=5):
            return df[col].groupby(df["instrument"]).transform(
                lambda s: s.rolling(d, min_periods=d).apply(
                    lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
                )
            )
        out["ideation_m_90"] = _mr("mci_a_mean") - _mr("oir_mean")

        # cgo
        ma60a = g["amount"].transform(lambda s: s.rolling(60, min_periods=30).mean())
        V = df["amount"] / (df["amount"] + ma60a + EPS)
        # path RP
        cgo_vals = []
        for _, sub in df.groupby("instrument", sort=False):
            px = sub["day_close"].to_numpy(dtype=float)
            v = V.loc[sub.index].to_numpy(dtype=float)
            n = len(px)
            out_c = np.full(n, np.nan)
            rp = np.nan
            for i in range(n):
                if not np.isfinite(px[i]) or px[i] <= 0:
                    continue
                if i == 0 or not np.isfinite(rp):
                    rp = px[i]
                else:
                    vv = v[i] if np.isfinite(v[i]) else 0.0
                    rp = vv * px[i - 1] + (1.0 - vv) * rp
                out_c[i] = (px[i] - rp) / px[i]
            cgo_vals.append(pd.Series(out_c, index=sub.index))
        cgo = pd.concat(cgo_vals).reindex(df.index)
        out["pv_cgo_neg"] = _attach_cs(df, cgo, neg=True)

        # vol_ul_shape
        r_on = df["day_open"] / df["pre_close"].replace(0, np.nan) - 1.0
        r_in = df["day_close"] / df["day_open"].replace(0, np.nan) - 1.0
        toi = r_on - r_in
        os_r = _attach_cs(df, df["open_share"], neg=False)
        toi_r = _attach_cs(df, toi, neg=False)
        raw_ul = os_r * (0.7 + 0.3 * toi_r)
        out["pv_vol_ul_shape_v3"] = _attach_cs(df, raw_ul, neg=False)

        # b97
        b_int = df["b_amount"] / df["volume"].replace(0, np.nan)
        s_int = df["s_amount"] / df["volume"].replace(0, np.nan)
        corr5 = []
        for _, sub in df.groupby("instrument", sort=False):
            corr5.append(b_int.loc[sub.index].rolling(5, min_periods=3).corr(s_int.loc[sub.index]))
        corr5 = pd.concat(corr5).reindex(df.index)
        tsrank = corr5.groupby(df["instrument"]).transform(
            lambda s: s.rolling(10, min_periods=5).apply(
                lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
            )
        )
        out["b97"] = -tsrank

        # m_88
        mci_tb = df["oir_mean"] / df["tot_bid_mean"].replace(0, np.nan)
        out["ideation_m_88"] = (
            mci_tb.groupby(df["instrument"]).transform(
                lambda s: s.rolling(5, min_periods=5).apply(
                    lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
                )
            )
            - df["tot_ask_mean"].groupby(df["instrument"]).transform(
                lambda s: s.rolling(5, min_periods=5).apply(
                    lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
                )
            )
        )

        # frag_shock
        deal_ma = g["deal_number"].transform(lambda s: s.rolling(60, min_periods=30).mean()).groupby(df["instrument"]).shift(1)
        vol_ma = g["volume"].transform(lambda s: s.rolling(60, min_periods=30).mean()).groupby(df["instrument"]).shift(1)
        frag = np.log(df["deal_number"] / (deal_ma + EPS)) - np.log(df["volume"] / (vol_ma + EPS))
        out["pv_frag_shock"] = _attach_cs(df, frag, neg=False)

        out["pv_signed_deal"] = _attach_cs(df, df["signed_deal_imb"], neg=True)

        # deal_amihud
        dret = df["day_close"] / df["day_open"].replace(0, np.nan) - 1.0
        deal_ma2 = deal_ma
        ami = dret.abs() / np.sqrt(df["deal_number"] / (deal_ma2 + EPS))
        out["pv_deal_amihud"] = _attach_cs(df, ami, neg=True)

        # --- fin：A31 只留 fin_pv_cand09（drop e22/cand06/cand12 近复制）---
        mofi_r = _attach_cs(df, df["mofi_imb_mean"], neg=False)
        logs_r = _attach_cs(df, np.log1p(df["s_amount"].clip(lower=0)), neg=False)
        ar = df["accounts_receivable"] if "accounts_receivable" in df.columns else pd.Series(np.nan, index=df.index)
        ta = (
            df["total_assets"].replace(0, np.nan)
            if "total_assets" in df.columns
            else pd.Series(np.nan, index=df.index)
        )
        ar_ratio = ar / ta
        ar_delta = ar_ratio.groupby(df["instrument"]).diff()
        out["fin_pv_cand09"] = _attach_cs(
            df,
            (mofi_r - logs_r) * (0.7 + 0.3 * _attach_cs(df, ar_delta, neg=False)),
            neg=False,
        )

        # cleanup placeholders / ensure all cols
        for c in FEATURE_COLS:
            if c not in out.columns:
                out[c] = np.nan
            out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

        out["close"] = pd.to_numeric(df["day_close"], errors="coerce")
        return out[["date", "instrument", "close"] + FEATURE_COLS]


    def build_selected33_panel(datasources, start_date, end_date) -> pd.DataFrame:
        """Fat panel once + 33 pandas legs."""
        t0 = time.time()
        bar1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")
        financial = datasources.get("financial", "bigalpha_2026_financial")

        pv = query_fat_panel(bar1m, start_date, end_date)
        fin = query_fin_panel(financial, start_date, end_date)
        cal = pv[["date", "instrument"]].drop_duplicates()
        fin_ttm = _sparse_pit_ffill(cal, fin, ["net_cffoa"], "ttm")
        fin_lf = _sparse_pit_ffill(
            cal, fin, ["moneytary_assets", "accounts_receivable", "total_assets"], "lf"
        )
        flt = query_fltcap(start_date, end_date)

        panel = pv.merge(fin_ttm, on=["date", "instrument"], how="left")
        panel = panel.merge(fin_lf, on=["date", "instrument"], how="left")
        panel = panel.merge(flt, on=["date", "instrument"], how="left")

        wide = compute_all_legs(panel)
        sd, ed = pd.to_datetime(start_date).normalize(), pd.to_datetime(end_date).normalize()
        wide = wide[(wide["date"] >= sd) & (wide["date"] <= ed)].reset_index(drop=True)
        logger.info(
            "selected33 wide ready",
            rows=len(wide),
            days=int(wide["date"].nunique()),
            sec=round(time.time() - t0, 1),
        )
        return wide


    def build_selected_features(bar1m_table, financial_table, sd, ed):
        """胖面板 + 30 pandas 腿；用 panel.day_close 作 close/next_ret（不再二次全窗 query）。"""
        ds = {"bar1m": bar1m_table, "financial": financial_table}
        out = build_selected33_panel(ds, sd, ed)
        if out.empty or "close" not in out.columns:
            raise RuntimeError(f"特征面板为空: {sd}~{ed} table={bar1m_table}")
        out = out.sort_values(["instrument", "date"]).reset_index(drop=True)
        out["next_ret"] = out.groupby("instrument", sort=False)["close"].shift(-1) / out["close"] - 1.0
        sd_ts, ed_ts = pd.to_datetime(sd).normalize(), pd.to_datetime(ed).normalize()
        out = out[(out["date"] >= sd_ts) & (out["date"] <= ed_ts)].reset_index(drop=True)
        logger.info("特征构建完成", rows=len(out), n_feat=len(FEATURE_COLS))
        return out

    class TinyTransformer(nn.Module):
        def __init__(self, input_size, num_classes):
            super().__init__()
            self.embedding = nn.Linear(input_size, D_MODEL)
            self.register_buffer(
                "positional_encoding",
                sinusoidal_positional_encoding(SEQ_LEN, D_MODEL),
                persistent=False,
            )
            layer = nn.TransformerEncoderLayer(
                d_model=D_MODEL,
                nhead=NHEAD,
                dim_feedforward=DIM_FF,
                dropout=DROPOUT,
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
            self.fc = nn.Linear(D_MODEL, num_classes)

        def forward(self, x):
            h = self.embedding(x) + self.positional_encoding[:, : x.size(1), :]
            h = self.encoder(h)
            return self.fc(h[:, -1, :])

    class StockDataset(Dataset):
        def __init__(self, arr_x, arr_y, seq_len):
            self.x = torch.tensor(arr_x, dtype=torch.float32)
            self.y = torch.tensor(arr_y, dtype=torch.long)
            self.seq_len = seq_len

        def __getitem__(self, index):
            s_end = index + self.seq_len
            return self.x[index:s_end], self.y[s_end - 1]

        def __len__(self):
            return len(self.x) - self.seq_len + 1

    class TestStockDataset(Dataset):
        def __init__(self, arr_x, seq_len):
            self.x = torch.tensor(arr_x, dtype=torch.float32)
            self.seq_len = seq_len

        def __getitem__(self, index):
            return self.x[index : index + self.seq_len]

        def __len__(self):
            return len(self.x) - self.seq_len + 1

    def _scale_train(df):
        mu = df[FEATURE_COLS].mean()
        sd = df[FEATURE_COLS].std().replace(0, 1.0).fillna(1.0)
        out = df.copy()
        out[FEATURE_COLS] = (out[FEATURE_COLS] - mu) / sd
        return out, mu, sd

    def _scale_apply(df, mu, sd):
        out = df.copy()
        for c in FEATURE_COLS:
            out[c] = out[c].fillna(0.5)
        out[FEATURE_COLS] = (out[FEATURE_COLS] - mu) / sd
        return out

    def _make_train_ds(df):
        d = df.dropna(subset=FEATURE_COLS + ["label"]).sort_values("date")
        if len(d) < SEQ_LEN:
            return None
        return StockDataset(
            d[FEATURE_COLS].to_numpy(np.float32),
            d["label"].to_numpy(np.int64),
            SEQ_LEN,
        )

    def _make_test_ds(df):
        d = df.sort_values("date")
        # 推理允许特征缺失填 0（仅序列输入；输出仍按有效行对齐）
        x = d[FEATURE_COLS].fillna(0.0).to_numpy(np.float32)
        if len(d) < SEQ_LEN:
            return None, d
        return TestStockDataset(x, SEQ_LEN), d

    # ---------- 1) 训练集（写死表）----------
    logger.info("构建训练集", start=TRAIN_START, end=TRAIN_END)
    train_raw = build_selected_features(TRAIN_BAR1M, TRAIN_FIN, TRAIN_START, TRAIN_END)
    train_raw["label"] = train_raw.groupby("date")["next_ret"].transform(_ntile10)
    train_raw = train_raw.dropna(subset=["label"])
    # 截面秩缺失填中性 0.5，避免单腿全 NaN 清空样本
    for c in FEATURE_COLS:
        train_raw[c] = train_raw[c].fillna(0.5)
    train_sc, feat_mu, feat_sd = _scale_train(train_raw)

    groups = [g for _, g in train_sc.groupby("instrument", sort=False)]
    with ThreadPoolExecutor() as ex:
        tds = [ds for ds in ex.map(_make_train_ds, groups) if ds is not None and len(ds) > 0]
    if not tds:
        raise RuntimeError("无训练序列样本")
    train_loader = DataLoader(
        ConcatDataset(tds), batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )

    # ---------- 2) 训练 TinyTransformer ----------
    model = TinyTransformer(INPUT_SIZE, NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    weights = torch.tensor(
        [10, 2.5, 1.33, 1.25, 1, 1, 1.25, 1.33, 2.5, 10], dtype=torch.float32, device=DEVICE
    )
    crit = nn.CrossEntropyLoss(weight=weights)
    t_fit = time.time()
    logger.info("开始训练", samples=len(train_loader.dataset), device=str(DEVICE))
    model.train()
    for epoch in range(NUM_EPOCHS):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
    logger.info("训练完成", sec=round(time.time() - t_fit, 1))

    # ---------- 3) 测试集（datasources）----------
    bar1m_table = datasources["bar1m"]
    financial_table = datasources["financial"]
    # 序列回看：特征函数已含 LOOKBACK；再扩 SEQ 保证首日有序列
    test_start = (
        pd.to_datetime(start_date) - pd.Timedelta(days=SEQ_LEN * 3 + 40)
    ).strftime("%Y-%m-%d 00:00:00")
    logger.info("构建测试集", bar1m=bar1m_table, start=test_start, end=end_date)
    test_raw = build_selected_features(bar1m_table, financial_table, test_start, end_date)
    test_sc = _scale_apply(test_raw, feat_mu, feat_sd)

    score_w = torch.tensor(
        [-10, -2.5, -1.33, -1.25, -1, 1, 1.25, 1.33, 2.5, 10], device=DEVICE
    )
    rows = []
    model.eval()
    with torch.no_grad():
        for _, g in test_sc.groupby("instrument", sort=False):
            ds, dmeta = _make_test_ds(g)
            if ds is None:
                continue
            loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
            preds = []
            for xb in loader:
                xb = xb.to(DEVICE)
                prob = torch.softmax(model(xb), dim=1)
                preds.append(torch.sum(prob * score_w, dim=1).cpu().numpy())
            pred = np.concatenate(preds)
            # 序列末对齐：丢掉前 SEQ_LEN-1 行
            meta = dmeta.iloc[SEQ_LEN - 1 :].copy()
            if len(meta) != len(pred):
                n = min(len(meta), len(pred))
                meta = meta.iloc[:n]
                pred = pred[:n]
            meta = meta[["date", "instrument"]].copy()
            meta["factor"] = pred
            rows.append(meta)

    if not rows:
        raise RuntimeError("测试推理无有效输出")
    pred_df = pd.concat(rows, ignore_index=True)
    sd0, ed0 = pd.to_datetime(start_date), pd.to_datetime(end_date)
    pred_df = pred_df[(pred_df["date"] >= sd0) & (pred_df["date"] <= ed0)]

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()
    pred_df["date"] = pd.to_datetime(pred_df["date"]).dt.normalize()
    pred_df["instrument"] = pred_df["instrument"].astype(str)

    result = pd.merge(pred_df, stk_pool, how="inner", on=["date", "instrument"])
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result = (
        result.dropna(subset=["factor"])
        .drop_duplicates(["date", "instrument"])
        .reset_index(drop=True)[["date", "instrument", "factor"]]
    )
    logger.info("因子完成", rows=len(result))
    return result


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()
    M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
        start_date="2024-01-01",
        end_date="2024-12-31",
    )
